# Predictive Maintenance for Manufacturing
## Cost-Sensitive Machine Failure Prediction

### Business Objective
제조 설비의 센서 데이터를 활용하여 설비 고장 확률을 예측하고,
단순 Accuracy 최대화가 아니라 **고장 미탐지(False Negative) 비용과
불필요한 점검(False Positive) 비용의 trade-off를 고려한 유지보수 의사결정 시스템**을 구축한다.

### 분석 목표
1. 심각한 클래스 불균형을 가진 고장 데이터를 안정적으로 모델링
2. 여러 불균형 처리 전략 및 모델 성능 비교
3. PR-AUC, Recall 중심으로 고장 탐지 성능 평가
4. 비즈니스 비용을 반영한 최적 Threshold 탐색
5. Feature Importance / SHAP을 통한 주요 고장 위험 요인 해석
6. 예측 확률을 유지보수 우선순위 정책으로 변환

## 1. Library & Data Loading

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

RANDOM_STATE = 42

df = pd.read_csv("ai4i2020.csv")
df.head()

## 2. Data Understanding

In [ ]:
print("Shape:", df.shape)
print("\nData Types")
print(df.dtypes)

print("\nMissing Values")
print(df.isnull().sum())

print("\nTarget Distribution")
print(df["Machine failure"].value_counts())

print("\nTarget Ratio (%)")
print(df["Machine failure"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
failure_rate = df["Machine failure"].mean()

plt.figure(figsize=(6, 4))
df["Machine failure"].value_counts().sort_index().plot(kind="bar")
plt.title(f"Machine Failure Distribution | Failure Rate = {failure_rate:.2%}")
plt.xlabel("Machine Failure")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

### Observation
고장 데이터는 전체의 약 3.4% 수준으로 매우 희소하다.

따라서 Accuracy만으로 모델을 평가하면 정상 설비를 대부분 정상으로 예측하는 모델도
높은 점수를 받을 수 있다.

본 프로젝트에서는 **Recall, Precision, F1, ROC-AUC와 함께 PR-AUC를 핵심 지표로 사용한다.**

## 3. Leakage Review & Feature Definition

In [ ]:
# 식별자 변수 제거
drop_id_cols = ["UDI", "Product ID"]

# 개별 고장 유형 변수
# 최종 Machine failure와 직접적으로 연결되는 사후/세부 고장 정보이므로
# 실제 조기예측 상황을 가정하여 feature에서 제외한다.
failure_mode_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]

target = "Machine failure"

model_df = df.drop(columns=drop_id_cols)

X = model_df.drop(columns=[target] + failure_mode_cols)
y = model_df[target]

print("Features:")
print(X.columns.tolist())

print("\nX shape:", X.shape)
print("y shape:", y.shape)

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Failure rate:", round(y_train.mean(), 4))
print("Test :", X_test.shape, "Failure rate:", round(y_test.mean(), 4))

## 5. Preprocessing Design

전처리는 반드시 Train 데이터에서만 학습되도록 Pipeline 내부에서 수행한다.

- Temperature 변수: StandardScaler → PCA 1 component
- Numeric 변수: 그대로 사용
- Type 변수: One-Hot Encoding
- Test 데이터에는 Train에서 학습한 변환만 적용

이 구조를 통해 전처리 단계의 Data Leakage를 방지한다.

In [ ]:
temperature_features = [
    "Air temperature [K]",
    "Process temperature [K]"
]

numeric_features = [
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

categorical_features = ["Type"]

temperature_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("temp_pca", temperature_pipeline, temperature_features),
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
    ],
    remainder="drop"
)

preprocessor

## 6. Baseline vs Imbalance Strategies

In [ ]:
def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob)
    }


experiments = {
    "RF Baseline": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "RF Class Weight": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "RF SMOTE": ImbPipeline([
        ("prep", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
}

results = []
fitted_experiments = {}

for name, model in experiments.items():
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    metrics = evaluate_predictions(y_test, pred, prob)
    metrics["Model"] = name

    results.append(metrics)
    fitted_experiments[name] = model

results_df = (
    pd.DataFrame(results)
      .set_index("Model")
      .sort_values("PR-AUC", ascending=False)
)

results_df.style.format("{:.3f}")

### Interpretation Guide
불균형 데이터에서는 Accuracy보다 아래 지표를 우선해서 본다.

- **Recall**: 실제 고장 중 얼마나 탐지했는가
- **Precision**: 고장 알람 중 실제 고장이 얼마나 되는가
- **PR-AUC**: 희소한 positive class에 대한 전반적인 분류 품질
- **ROC-AUC**: 전체 임계값 영역에서의 ranking 성능

SMOTE가 반드시 정답은 아니다.
Baseline / Class Weight / SMOTE를 실험적으로 비교하고 가장 적합한 전략을 선택한다.

## 7. Multi-Model Benchmark

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),
    "LightGBM": LGBMClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        verbose=-1
    )
}

benchmark_results = []
benchmark_models = {}

for name, estimator in models.items():

    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", estimator)
    ])

    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]

    metrics = evaluate_predictions(y_test, pred, prob)
    metrics["Model"] = name

    benchmark_results.append(metrics)
    benchmark_models[name] = pipe

benchmark_df = (
    pd.DataFrame(benchmark_results)
      .set_index("Model")
      .sort_values("PR-AUC", ascending=False)
)

benchmark_df.style.format("{:.3f}")

## 8. Precision-Recall Curve

In [ ]:
plt.figure(figsize=(8, 6))

for name, model in benchmark_models.items():
    prob = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)

    plt.plot(recall, precision, label=f"{name} | AP={ap:.3f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 9. Select Champion Model

In [ ]:
champion_name = benchmark_df["PR-AUC"].idxmax()
champion_model = benchmark_models[champion_name]

print("Champion Model:", champion_name)
display(benchmark_df.loc[[champion_name]])

Champion 모델은 단순 Accuracy가 아니라 **PR-AUC 중심으로 선정**한다.

다만 최종 운영 모델 선택에서는 PR-AUC뿐 아니라
Recall, Precision 및 비즈니스 비용까지 함께 고려한다.

## 10. Baseline Threshold = 0.50

In [ ]:
y_prob = champion_model.predict_proba(X_test)[:, 1]

baseline_threshold = 0.50
y_pred_05 = (y_prob >= baseline_threshold).astype(int)

print(classification_report(y_test, y_pred_05, digits=3))

cm_05 = confusion_matrix(y_test, y_pred_05)
print("Confusion Matrix")
print(cm_05)

## 11. Cost-Sensitive Threshold Optimization

In [ ]:
# 예시 비즈니스 비용
# 실제 기업에서는 생산중단 비용, 정비 인건비, 부품비 등의 실제 데이터를 사용해야 한다.

FN_COST = 10000   # 실제 고장을 놓친 경우
FP_COST = 500     # 정상 설비를 불필요하게 점검한 경우

thresholds = np.arange(0.01, 1.00, 0.01)

cost_records = []

for threshold in thresholds:
    pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    total_cost = (fn * FN_COST) + (fp * FP_COST)

    cost_records.append({
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Recall": recall_score(y_test, pred, zero_division=0),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Total Cost": total_cost
    })

cost_df = pd.DataFrame(cost_records)

best_row = cost_df.loc[cost_df["Total Cost"].idxmin()]

print("Optimal Threshold")
display(best_row.to_frame().T)

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(cost_df["Threshold"], cost_df["Total Cost"])
plt.axvline(
    best_row["Threshold"],
    linestyle="--",
    label=f'Optimal threshold = {best_row["Threshold"]:.2f}'
)
plt.xlabel("Decision Threshold")
plt.ylabel("Expected Cost")
plt.title("Business Cost by Classification Threshold")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 12. Final Model Evaluation at Business Threshold

In [ ]:
optimal_threshold = float(best_row["Threshold"])

y_pred_optimal = (y_prob >= optimal_threshold).astype(int)

print(f"Optimal Threshold: {optimal_threshold:.2f}")
print(classification_report(y_test, y_pred_optimal, digits=3))

cm_opt = confusion_matrix(y_test, y_pred_optimal)
print("Confusion Matrix")
print(cm_opt)

baseline_metrics = evaluate_predictions(y_test, y_pred_05, y_prob)
optimal_metrics = evaluate_predictions(y_test, y_pred_optimal, y_prob)

comparison = pd.DataFrame(
    [baseline_metrics, optimal_metrics],
    index=["Threshold 0.50", f"Threshold {optimal_threshold:.2f}"]
)

comparison.style.format("{:.3f}")

## 13. Threshold Trade-off

In [ ]:
selected_thresholds = [0.10, 0.20, 0.30, 0.40, 0.50]

rows = []

for t in selected_thresholds:
    pred = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    rows.append({
        "Threshold": t,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "False Positive": fp,
        "False Negative": fn,
        "Expected Cost": fn * FN_COST + fp * FP_COST
    })

threshold_comparison = pd.DataFrame(rows)

threshold_comparison.style.format({
    "Precision": "{:.3f}",
    "Recall": "{:.3f}",
    "Expected Cost": "${:,.0f}"
})

## 14. Feature Importance

In [ ]:
# Tree 기반 champion model에서 feature importance 확인
estimator = champion_model.named_steps["model"]

if hasattr(estimator, "feature_importances_"):
    feature_names = champion_model.named_steps["prep"].get_feature_names_out()

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": estimator.feature_importances_
    }).sort_values("Importance", ascending=False)

    display(importance_df)

    plt.figure(figsize=(8, 5))
    top_imp = importance_df.head(10).sort_values("Importance")
    plt.barh(top_imp["Feature"], top_imp["Importance"])
    plt.title(f"Feature Importance | {champion_name}")
    plt.xlabel("Importance")
    plt.show()
else:
    print("Champion model does not expose feature_importances_.")

## 15. Optional: SHAP Analysis

SHAP이 설치되어 있지 않다면 먼저 다음 명령을 한 번 실행한다.

```python
!pip install shap
```

In [ ]:
# 선택 실행 셀
try:
    import shap

    estimator = champion_model.named_steps["model"]
    X_test_transformed = champion_model.named_steps["prep"].transform(X_test)

    if hasattr(X_test_transformed, "toarray"):
        X_test_transformed = X_test_transformed.toarray()

    feature_names = champion_model.named_steps["prep"].get_feature_names_out()

    if hasattr(estimator, "feature_importances_"):
        explainer = shap.TreeExplainer(estimator)
        shap_values = explainer.shap_values(X_test_transformed)

        # sklearn / shap 버전에 따라 반환 형태가 다를 수 있음
        if isinstance(shap_values, list):
            values = shap_values[-1]
        else:
            values = shap_values

        shap.summary_plot(
            values,
            X_test_transformed,
            feature_names=feature_names
        )
    else:
        print("Tree SHAP is not available for the selected model.")

except ImportError:
    print("SHAP is not installed. Run: pip install shap")
except Exception as e:
    print("SHAP 실행 중 버전 호환 문제가 발생했습니다:", e)

## 16. Maintenance Risk Policy

In [ ]:
prediction_result = X_test.copy()

prediction_result["Actual Failure"] = y_test.values
prediction_result["Failure Probability"] = y_prob

def assign_risk_level(prob):
    if prob < 0.15:
        return "Normal"
    elif prob < optimal_threshold:
        return "Watch"
    elif prob < 0.60:
        return "Preventive Inspection"
    else:
        return "Critical"

prediction_result["Maintenance Action"] = (
    prediction_result["Failure Probability"]
    .apply(assign_risk_level)
)

prediction_result[
    ["Failure Probability", "Maintenance Action", "Actual Failure"]
].sort_values(
    "Failure Probability",
    ascending=False
).head(20)

In [ ]:
policy_summary = (
    prediction_result
    .groupby("Maintenance Action")
    .agg(
        Machines=("Actual Failure", "size"),
        Actual_Failures=("Actual Failure", "sum"),
        Avg_Risk=("Failure Probability", "mean")
    )
    .sort_values("Avg_Risk", ascending=False)
)

policy_summary

## 17. Business Conclusion

### Key Findings
- 전체 데이터에서 실제 고장은 매우 희소하므로 Accuracy 단독 평가는 부적절하다.
- 여러 불균형 처리 전략과 모델을 동일한 Test set에서 비교하였다.
- 최종 모델은 PR-AUC를 중심으로 선정하고 Recall / Precision trade-off를 함께 평가하였다.
- 기본 0.5 threshold를 고정하지 않고 False Negative와 False Positive의 비즈니스 비용을 반영하여 최적 threshold를 탐색하였다.
- 최종 예측 확률을 유지보수 단계로 변환하여 모델 결과를 실제 운영 의사결정과 연결하였다.

### Operational Recommendation
예측 모델을 단순 binary classifier로 사용하는 대신,
설비별 failure probability를 기반으로 점검 우선순위를 관리하는
**Risk-Based Predictive Maintenance System**으로 운영하는 것이 적절하다.

### Limitation
본 프로젝트의 FN / FP 비용은 예시 가정값이다.
실제 기업 적용 시에는 생산라인 중단 손실, 정비 인건비, 부품 교체비,
설비별 중요도 및 정비 가능 인력 등을 반영하여 비용 함수를 재설계해야 한다.

## 18. Portfolio Summary

**Cost-Sensitive Predictive Maintenance for Manufacturing**

센서 기반 설비 고장 예측 문제에서 약 3.4%의 희소한 고장 데이터에 대응하기 위해
Baseline, Class Weight, SMOTE 전략과 다수의 분류 모델을 비교하였다.

Accuracy 대신 Recall 및 PR-AUC를 중심으로 평가하고,
고장 미탐지 비용과 불필요한 점검 비용을 반영한 Cost-Sensitive Threshold Optimization을 통해
운영 의사결정 기준을 설계하였다.

또한 Feature Importance / SHAP을 활용하여 주요 고장 위험 요인을 해석하고,
예측 확률을 Normal, Watch, Preventive Inspection, Critical 단계로 분류하여
유지보수팀의 점검 우선순위 결정에 활용할 수 있는 의사결정 시스템으로 확장하였다.